# 06 · Looking inside: attention maps (paper Figs. 3 & 6)

> **Paper:** §4.2 "Ablations", Fig. 3 (encoder self-attention), Fig. 6 (decoder attention)

We've traced every tensor. Now let's see what the model actually *looks at*. We'll reproduce two figures from the paper on our own image, using PyTorch **forward hooks** — the standard tool for extracting intermediate values without editing the model.

The finding, in one line: **the encoder separates instances; the decoder then traces their edges.**

## 0. Setup — everything this notebook needs

This notebook is **self-contained**: only third-party packages are imported, and every DETR-specific piece is written out below. Nothing comes from this repo, so you can read straight through without chasing a helper into another file.

Run this section once, then forget about it.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names and plot colors

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

DETR predicts boxes as **`cxcywh`** — centre + size, normalized to `[0, 1]`. IoU and plotting want **`xyxy`** corners. Mixing the two up is the single most common bug in detection code, so both conversions live here.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

### Images in, tensors out

DETR's eval transform resizes the shortest side to 800px and ImageNet-normalizes. There is no fixed crop — the model accepts any input size.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

### Drawing detections

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

### DETR itself

Backbone, positional encoding, transformer and prediction heads, written out. This is the same architecture as [`models/`](../models) with the inference path kept.

The proof that it is faithful is `load_state_dict(...)` below: it is **strict**, so every parameter name here has to match Facebook's released checkpoint exactly or it raises.

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(),
                                       PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries

    def forward(self, images, mask=None):
        if mask is None:
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)
        return {"pred_logits": self.class_embed(hs)[-1],
                "pred_boxes": self.bbox_embed(hs).sigmoid()[-1]}


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR()
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep

In [ ]:
device = get_device()
model = load_pretrained_detr(device=device)
im = load_image("cats")
print("device:", device, "| image:", im.size)

## 1. Capturing attention with forward hooks

`nn.MultiheadAttention` returns `(output, attention_weights)`, but DETR's code keeps only `[0]` and throws the weights away. A **forward hook** intercepts the return value so we can keep both — no changes to `models/transformer.py` required.

In [ ]:
conv_features, enc_attn, dec_attn = [], [], []

# A forward hook fires every time a module produces output. We use them to grab
# tensors from inside the model without changing a single line of the model itself.
hooks = [
    # the ResNet trunk: we want the feature-map size, to fold 850 tokens back into a grid
    model.backbone[0].register_forward_hook(
        lambda m, i, o: conv_features.append(o[0].shape)),
    # last ENCODER layer: image tokens attending to image tokens
    model.transformer.encoder.layers[-1].self_attn.register_forward_hook(
        lambda m, i, o: enc_attn.append(o[1])),
    # last DECODER layer: object queries attending to the encoder memory
    model.transformer.decoder.layers[-1].multihead_attn.register_forward_hook(
        lambda m, i, o: dec_attn.append(o[1])),
]

with torch.no_grad():
    outputs = model(default_transform(im).unsqueeze(0).to(device))

for h in hooks:
    h.remove()          # ALWAYS remove hooks -- they leak memory and slow later runs

_, _, Hf, Wf = conv_features[0]
enc = enc_attn[0][0].cpu()          # shape: (850, 850)   token -> token
dec = dec_attn[0][0].cpu()          # shape: (100, 850)   query -> token

print(f"feature grid      : {Hf} x {Wf} = {Hf*Wf} tokens")
print(f"encoder attention : {tuple(enc.shape)}  every token's view of every other token")
print(f"decoder attention : {tuple(dec.shape)}  every query's view of the image")
print(f"\nrows are softmax distributions -> each sums to 1: {float(enc[0].sum()):.4f}")

## 2. Encoder self-attention (Fig. 3)

Pick a few **reference points** in the image. For each, show which other pixels its token attends to. Paper's caption: *"The encoder is able to separate individual instances."*

In [ ]:
# reference points as (row, col) in the FEATURE grid, not pixels
ref_points = [(int(Hf*0.55), int(Wf*0.25)),    # left cat
              (int(Hf*0.35), int(Wf*0.78)),    # right cat
              (int(Hf*0.18), int(Wf*0.14)),    # left remote
              (int(Hf*0.90), int(Wf*0.50))]    # couch / background

fig, axes = plt.subplots(2, len(ref_points)+1, figsize=(16, 6),
                         gridspec_kw={"height_ratios": [1, 1]})
gs = axes[0, 0].get_gridspec()
for ax in axes[:, 0]:
    ax.remove()
axbig = fig.add_subplot(gs[:, 0])
axbig.imshow(im); axbig.axis("off"); axbig.set_title("reference points")

for k, (r, c) in enumerate(ref_points):
    # convert feature-grid coords -> pixel coords for the marker
    axbig.plot(c / Wf * im.size[0], r / Hf * im.size[1], "o",
               color=COLORS[k], markersize=13, markeredgecolor="w", markeredgewidth=2)

for k, (r, c) in enumerate(ref_points):
    idx = r * Wf + c                       # flatten (row, col) -> token index
    a = enc[idx].reshape(Hf, Wf)           # that token's attention over the grid
    ax = axes[0, k+1]
    ax.imshow(a, cmap="viridis"); ax.axis("off")
    ax.set_title(f"self-attention ({r},{c})", fontsize=9, color=COLORS[k])
    ax2 = axes[1, k+1]
    ax2.imshow(im)
    ax2.imshow(torch.nn.functional.interpolate(
        a[None, None], size=(im.size[1], im.size[0]), mode="bilinear")[0, 0],
        cmap="viridis", alpha=0.65)
    ax2.axis("off"); ax2.set_title("overlaid", fontsize=9)
plt.tight_layout(); plt.show()

Look at what happened: a point on the **left cat** lights up the left cat *and essentially nothing else* — not the right cat, even though both are cats with near-identical texture. The encoder has already grouped pixels into **instances**, before any box is predicted.

This is why removing the encoder costs 3.9 AP overall and **6.0 AP on large objects** (Table 2). A large object spans dozens of grid cells; only global self-attention can bind them into one thing.

> *"The encoder seems to separate instances already, which likely simplifies object extraction and localization for the decoder."* — §4.2

## 3. Decoder cross-attention (Fig. 6)

Now the other half: for each *detected object*, which pixels did its query read from?

In [ ]:
probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # (queries, classes) = (100, 91)
keep = probs.max(-1).values > 0.9
kept_idx = keep.nonzero().flatten().tolist()
boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), im.size)

print(f"{len(kept_idx)} detections, from query slots {kept_idx}")
for q, p in zip(kept_idx, probs[keep]):
    print(f"   query {q:3d}: {COCO_CLASSES[p.argmax()]:<8} {p.max():.3f}")

In [ ]:
n = len(kept_idx)
fig, axes = plt.subplots(2, n, figsize=(3.4*n, 7))
axes = axes.reshape(2, n)

for j, (q, box) in enumerate(zip(kept_idx, boxes)):
    a = dec[q].reshape(Hf, Wf)                 # this query's attention over the image
    axes[0, j].imshow(a, cmap="cividis"); axes[0, j].axis("off")
    lbl = COCO_CLASSES[probs[q].argmax()]
    axes[0, j].set_title(f"query {q} -> {lbl}", fontsize=10)

    axes[1, j].imshow(im)
    axes[1, j].imshow(torch.nn.functional.interpolate(
        a[None, None], size=(im.size[1], im.size[0]), mode="bilinear")[0, 0],
        cmap="cividis", alpha=0.7)
    x0, y0, x1, y1 = box.tolist()
    axes[1, j].add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0,
                          fill=False, color="w", lw=2.5))
    axes[1, j].axis("off"); axes[1, j].set_title("with predicted box", fontsize=9)

plt.suptitle("Decoder cross-attention: each query attends to ONE object's extremities", y=1.0)
plt.tight_layout(); plt.show()

Two things to notice:

1. **Each query attends to exactly one object.** Query 61's map covers the left cat only; query 98's covers the right cat only. This is the set prediction from notebook `05`, visible in the attention: one query, one object.
2. **Attention concentrates on edges and extremities** — paws, ear tips, the ends of the remotes. Paper: *"decoder attention is fairly local, meaning that it mostly attends to object extremities such as heads or legs."*

That division of labor is the whole architecture in a sentence: the encoder does the **global** work of separating instances; the decoder does the **local** work of finding each one's boundary, which is precisely what you need to regress a box.

## 4. What do the ∅ queries look at?

95 queries predicted "no object". Are they attending to nothing, or somewhere useless?

In [ ]:
empty_idx = (~keep).nonzero().flatten()[:4].tolist()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, q in zip(axes, empty_idx):
    ax.imshow(im)
    a = dec[q].reshape(Hf, Wf)
    ax.imshow(torch.nn.functional.interpolate(
        a[None, None], size=(im.size[1], im.size[0]), mode="bilinear")[0, 0],
        cmap="cividis", alpha=0.7)
    ax.axis("off")
    ax.set_title(f"query {q}  (∅, p={probs[q].max():.2f})", fontsize=9)
plt.suptitle("Queries that predict 'no object' still attend somewhere — usually background")
plt.tight_layout(); plt.show()

They still attend somewhere (attention rows must sum to 1 — a softmax can't abstain). But the content they gather doesn't produce a confident class, so `class_embed` routes them to `∅`. The query is "asking its question" and getting the answer *"nothing here."*

## 5. Attention sharpens across decoder layers

Capture the cross-attention at every layer and watch one object's map tighten.

In [ ]:
per_layer = []
hooks = [l.multihead_attn.register_forward_hook(
            lambda m, i, o: per_layer.append(o[1][0].cpu()))
         for l in model.transformer.decoder.layers]
with torch.no_grad():
    _ = model(default_transform(im).unsqueeze(0).to(device))
for h in hooks: h.remove()

q = kept_idx[-1]
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for L, ax in enumerate(axes):
    ax.imshow(im)
    a = per_layer[L][q].reshape(Hf, Wf)
    ax.imshow(torch.nn.functional.interpolate(
        a[None, None], size=(im.size[1], im.size[0]), mode="bilinear")[0, 0],
        cmap="cividis", alpha=0.7)
    ax.axis("off"); ax.set_title(f"decoder layer {L+1}", fontsize=10)
plt.suptitle(f"Query {q}: attention narrowing onto its object, layer by layer")
plt.tight_layout(); plt.show()

Early layers attend broadly (the query hasn't found its object yet); later layers lock on. This is the visual counterpart of the AP curve in Fig. 4, where accuracy climbs +8.2 AP from decoder layer 1 to layer 6.

## What's next

`07` is the payoff: train DETR from scratch on a tiny dataset and watch the matching loss do its work.

## Exercises

**Exercise 1.** Which parts of the image does *everything* attend to? Compute the column sums of `enc`.

<details><summary>Solution</summary>

```python
popularity = enc.sum(0).reshape(Hf, Wf)      # column sum = how much each token is attended TO
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(popularity, cmap="magma"); ax[0].set_title("attention received per token")
ax[1].imshow(im)
ax[1].imshow(torch.nn.functional.interpolate(
    popularity[None, None], size=(im.size[1], im.size[0]), mode="bilinear")[0, 0],
    cmap="magma", alpha=.7)
for a in ax: a.axis("off")
plt.show()
```

Recall from notebook `01`: **rows** of an attention matrix sum to 1, columns do not. So `enc.sum(0)` measures how "popular" each token is as a source of information — typically object regions and high-contrast boundaries, not flat background.
</details>

---

**Exercise 2.** Extract **per-head** attention instead of the head-average.

<details><summary>Solution</summary>

`nn.MultiheadAttention` averages heads by default. Ask for them separately:

```python
per_head = []
def hook(m, i, o): pass
h = model.transformer.decoder.layers[-1].multihead_attn
orig = h.forward
def patched(*a, **kw):
    kw["need_weights"] = True; kw["average_attn_weights"] = False
    return orig(*a, **kw)
h.forward = patched
store = []
hk = h.register_forward_hook(lambda m, i, o: store.append(o[1][0].cpu()))
with torch.no_grad(): _ = model(default_transform(im).unsqueeze(0).to(device))
hk.remove(); h.forward = orig

heads = store[0]                       # (nheads, 100, 850)
print("per-head attention:", tuple(heads.shape))
q = kept_idx[0]
fig, axes = plt.subplots(1, 8, figsize=(18, 2.6))
for j, ax in enumerate(axes):
    ax.imshow(heads[j, q].reshape(Hf, Wf), cmap="cividis"); ax.axis("off")
    ax.set_title(f"head {j}", fontsize=8)
plt.show()
```

Different heads latch onto different parts of the same object — some the left edge, some the top. Averaging them (the default) blurs this, which is why notebook `06`'s maps look smoother than the paper's.
</details>

---

**Exercise 3.** Re-run the whole notebook on `load_image("street")`. Is instance separation as clean?

<details><summary>Solution</summary>

Change `im = load_image("cats")` in the first cell and re-run. Separation is noticeably **less** clean: the scene has many small objects, and at a 32×-downsampled grid a small object occupies barely one token — there simply aren't enough tokens to separate them.

This is the concrete reason behind the paper's weakest number. In Table 1, DETR and Faster RCNN-FPN+ tie on overall AP at **42.0** -- and at that same overall score DETR scores **20.5 AP_S** against Faster R-CNN's **26.6**. Six points of small-object AP, given away in exchange for the global attention that wins DETR its large-object lead (61.1 vs 53.4 AP_L).

(Watch the schedule when you read that table: the `+` rows are the long-schedule Faster R-CNN models, and those are the ones matched to DETR on overall AP. The short-schedule Faster RCNN-FPN scores 40.2 AP with 24.2 AP_S.)

Small objects are what DETR-DC5 (Exercise 2 of notebook `03`) and later Deformable DETR were designed to fix.
</details>

---

**Exercise 4.** Move `ref_points` onto the couch. Does the encoder group the whole couch or fragment it?

<details><summary>Solution</summary>

Try `(int(Hf*0.95), int(Wf*0.1))` and similar background points. The couch spans the entire image, and you'll typically see attention spread broadly across the background region rather than forming a tight blob.

Compare with the "couch" query's *decoder* attention in §3, which attended to the image **borders** — for an object filling the frame, the informative pixels are its edges, which are the frame edges.
</details>